# ARIMA Forecast – LastFM Sessions

**Exercise 3 – ML Challenge**

- Data processing: **PySpark** (mirrors Scala 3.5.1 logic exactly)
- Forecasting: **ARIMA(1,1,1)** via statsmodels

## 1. Start Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('LastFM_ARIMA') \
    .getOrCreate()

# Match Spark 3.5.1 (Scala) timestamp parsing behaviour
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'LEGACY')
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

## 2. Load data

In [ ]:
DATA_PATH = '../data/userid-timestamp-artid-artname-traid-traname.tsv'

plays = (
    spark.read
    .option('sep', '\t')
    .option('header', 'false')
    .option('timestampFormat', "yyyy-MM-dd'T'HH:mm:ss'Z'")
    .option('mode', 'DROPMALFORMED')
    .csv(DATA_PATH)
    .toDF('user_id', 'timestamp', 'artist_mb_id', 'artist_name', 'track_mb_id', 'track_name')
    .withColumn('timestamp', F.to_timestamp('timestamp', "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .filter(
        F.col('user_id').isNotNull() &
        F.col('timestamp').isNotNull() &
        F.col('track_name').isNotNull()
    )
    .select('user_id', 'timestamp', 'artist_name', 'track_name')
)

print(f'Total rows: {plays.count():,}')
plays.show(5, truncate=False)

## 3. Build sessions (gap > 20 min = new session)

In [ ]:
w_user = Window.partitionBy('user_id').orderBy('timestamp')

with_sessions = (
    plays
    .withColumn('prev_ts', F.lag('timestamp', 1).over(w_user))
    .withColumn(
        'new_session',
        F.when(
            F.col('prev_ts').isNull() |
            (F.unix_timestamp('timestamp') - F.unix_timestamp('prev_ts') > 20 * 60),
            1
        ).otherwise(0)
    )
    .withColumn('session_id', F.concat_ws('_', F.col('user_id'), F.sum('new_session').over(w_user)))
    .drop('prev_ts', 'new_session')
)

print(f'Total sessions: {with_sessions.select("session_id").distinct().count():,}')

## 4. Top 10 songs in the top 50 longest sessions by track count

In [ ]:
top50 = (
    with_sessions
    .groupBy('session_id')
    .count()
    .orderBy(F.desc('count'))
    .limit(50)
    .select('session_id')
)

top10 = (
    with_sessions
    .join(F.broadcast(top50), 'session_id')
    .groupBy('artist_name', 'track_name')
    .count()
    .orderBy(F.desc('count'))
    .limit(10)
)

print('Top 10 songs in top 50 longest sessions:')
top10.show(truncate=False)

## 5. Top 1 user by number of sessions

In [ ]:
top_user_row = (
    with_sessions
    .select('user_id', 'session_id').distinct()
    .groupBy('user_id')
    .count()
    .orderBy(F.desc('count'))
    .first()
)

top_user   = top_user_row['user_id']
top_n_sess = top_user_row['count']
print(f'Top user: {top_user}  -  {top_n_sess:,} sessions')

## 6. Weekly session count for the top user

In [ ]:
weekly_spark = (
    with_sessions
    .filter(F.col('user_id') == top_user)
    .groupBy('session_id')
    .agg(F.min('timestamp').alias('session_start'))
    .withColumn('week', F.date_trunc('week', 'session_start'))
    .groupBy('week')
    .agg(F.count('session_id').alias('n_sessions'))
    .orderBy('week')
)

weekly_spark.show(10)

## 7. Convert to pandas and plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weekly_pd = weekly_spark.toPandas()
weekly_pd['week'] = pd.to_datetime(weekly_pd['week']).dt.tz_localize(None)
weekly = weekly_pd.set_index('week')['n_sessions'].asfreq('W')

print(f'Date range: {weekly.index.min().date()} to {weekly.index.max().date()}')

weekly.plot(title=f'Weekly sessions - user {top_user}', figsize=(12, 4))
plt.ylabel('Sessions')
plt.tight_layout()
plt.show()

## 8. ARIMA(1,1,1) forecast - next 3 months (~13 weeks)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

model  = ARIMA(weekly.fillna(0), order=(1, 1, 1))
result = model.fit()
print(result.summary())

In [ ]:
FORECAST_WEEKS = 13

forecast = result.get_forecast(steps=FORECAST_WEEKS)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int()

fig, ax = plt.subplots(figsize=(14, 5))
weekly.plot(ax=ax, label='Historical', color='steelblue')
fc_mean.plot(ax=ax, label='Forecast', color='darkorange', linestyle='--')
ax.fill_between(
    fc_ci.index,
    fc_ci.iloc[:, 0],
    fc_ci.iloc[:, 1],
    color='darkorange', alpha=0.2, label='95% CI'
)
ax.set_title(f'ARIMA(1,1,1) - Weekly sessions forecast (next 3 months) - user {top_user}')
ax.set_ylabel('Sessions')
ax.legend()
plt.tight_layout()
plt.show()

print('Forecast values:')
fc_mean.rename('predicted_sessions').to_frame().join(fc_ci).round(2)

In [ ]:
spark.stop()